# Deepfake Detection — CRNN + Optuna Hyperparameter Optimization
ASVspoof 2019 LA dataset

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pytorch-optimizer optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 31.0 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import io
import os
import random
import pickle
from contextlib import redirect_stdout

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.functional as AF
import torchaudio.transforms as T
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, precision_score, recall_score, f1_score
)

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from google.colab import drive

In [ ]:
# ── Cell 3: Device & seed ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def set_seed(seed=22):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(22)

Using device: cuda


In [ ]:
# ── Cell 4: Mount Drive & load data ───────────────────────────────────────────
drive.mount('/content/drive')

DEV_PATH   = ('/content/drive/MyDrive/ASV_Final_Features_dev.pkl',  'dev')
EVAL_PATH  = ('/content/drive/MyDrive/ASV_Final_Features_eval.pkl', 'eval')
TRAIN_PATH = ('/content/drive/MyDrive/ASV_Final_Features_train.pkl','train')

def load_dataset(path):
    df = pd.read_pickle(path[0])
    print(f'DataFrame loaded for {path[1]} set. '
          f'Total samples: {len(df)}, shape: {df.shape}')
    return df

train_df = load_dataset(TRAIN_PATH)
dev_df   = load_dataset(DEV_PATH)
eval_df  = load_dataset(EVAL_PATH)

print('\nLabel distribution:')
print('Train:', train_df['label'].value_counts().to_dict())
print('Dev:  ', dev_df['label'].value_counts().to_dict())
print('Eval: ', eval_df['label'].value_counts().to_dict())

Mounted at /content/drive
DataFrame loaded for train set. Total samples: 25380, shape: (25380, 5)
DataFrame loaded for dev set. Total samples: 24844, shape: (24844, 5)
DataFrame loaded for eval set. Total samples: 71237, shape: (71237, 5)

Label distribution:
Train: {1: 22800, 0: 2580}
Dev:   {1: 22296, 0: 2548}
Eval:  {1: 63882, 0: 7355}


In [ ]:
# ── Cell 5: Prepare tensors ────────────────────────────────────────────────────
def prepare_tensors(df, feature_col='hybrid_features', label_col='label'):
    X = np.stack(df[feature_col].values).transpose(0, 2, 1).astype(np.float32)
    y = df[label_col].values
    return torch.from_numpy(X), torch.from_numpy(y)

X_train_tr, y_train_tr = prepare_tensors(train_df)
X_dev_tr,   y_dev_tr   = prepare_tensors(dev_df)
X_eval_tr,  y_eval_tr  = prepare_tensors(eval_df)

# Normalization stats
epsilon  = 1e-4
x_means  = X_train_tr.mean(dim=(0, 2), keepdim=True)
x_stds   = X_train_tr.std(dim=(0, 2),  keepdim=True) + epsilon

print(f'Train : {X_train_tr.shape}, {y_train_tr.shape}')
print(f'Dev   : {X_dev_tr.shape},   {y_dev_tr.shape}')
print(f'Eval  : {X_eval_tr.shape},  {y_eval_tr.shape}')

Train : torch.Size([25380, 128, 126]), torch.Size([25380])
Dev   : torch.Size([24844, 128, 126]),   torch.Size([24844])
Eval  : torch.Size([71237, 128, 126]),  torch.Size([71237])


In [ ]:
# ── Cell 6: Model components ───────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class AttentiveStatsPooling(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(128, input_dim, kernel_size=1),
            nn.Softmax(dim=2)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        weights = self.attention(x)
        mu = torch.sum(x * weights, dim=2)
        stdev = torch.sqrt(
            torch.sum(weights * (x ** 2), dim=2) - mu ** 2 + 1e-7
        )
        return torch.cat((mu, stdev), dim=1)

In [ ]:
# ── Cell 7: CRNN model (Optuna-parameterised) ──────────────────────────────────
class Deepfake_CRNN(nn.Module):
    """
    Parameters
    ----------
    means, stds       : normalization buffers (computed from train set)
    cnn_dropout       : Dropout2d rate after each conv block
    rnn_hidden_size   : LSTM hidden units
    rnn_num_layers    : number of LSTM layers
    rnn_dropout       : dropout between LSTM layers (only active if layers > 1)
    bidirectional     : whether to use a bidirectional LSTM
    """
    def __init__(self, means, stds,
                 cnn_dropout=0.2,
                 rnn_hidden_size=128,
                 rnn_num_layers=1,
                 rnn_dropout=0.0,
                 bidirectional=False):
        super().__init__()

        self.register_buffer('means', means.detach().clone().view(1, 128, 1))
        self.register_buffer('stds',  stds.detach().clone().view(1, 128, 1))

        # ── CNN backbone (3-channel: raw + delta + delta-delta) ────────────────
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            SEBlock(16),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(cnn_dropout),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            SEBlock(32),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(cnn_dropout),
        )

        # ── LSTM ───────────────────────────────────────────────────────────────
        # input_size = 32 filters * (128 freq / 4) = 32 * 32 = 1024
        self.bidirectional = bidirectional
        self.rnn = nn.LSTM(
            input_size=32 * 32,
            hidden_size=rnn_hidden_size,
            num_layers=rnn_num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=rnn_dropout if rnn_num_layers > 1 else 0.0,
        )

        # ── Attentive stats pooling ────────────────────────────────────────────
        rnn_out_dim = rnn_hidden_size * (2 if bidirectional else 1)
        self.pooling = AttentiveStatsPooling(input_dim=rnn_out_dim)

        # ── Classification head ────────────────────────────────────────────────
        # pooling output = rnn_out_dim * 2 (mean + std)
        self.fc = nn.Sequential(
            nn.Linear(rnn_out_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        # 1. Normalize + compute deltas
        x = (x - self.means) / (self.stds + 1e-7)
        delta1 = AF.compute_deltas(x)
        delta2 = AF.compute_deltas(delta1)
        x = torch.stack([x, delta1, delta2], dim=1)  # (B, 3, F, T)

        # 2. CNN
        x = self.conv_layers(x)                       # (B, 32, F', T')

        # 3. Reshape for LSTM
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()        # (B, T', 32, F')
        x = x.view(b, t, c * f)                       # (B, T', 1024)

        # 4. LSTM
        x, _ = self.rnn(x)                             # (B, T', H)

        # 5. Attentive pooling
        x = self.pooling(x)                            # (B, H*2)

        # 6. Classification
        return self.fc(x)                              # (B, 2)

In [ ]:
# ── Cell 8: Evaluation function ────────────────────────────────────────────────
def eval_model(model, dl, device, dataset_name='Dataset', show_plots=True):
    """
    Returns (eer, fig).
    Pass show_plots=False during Optuna trials to suppress all output.
    """
    model.eval()
    all_labels, all_preds, all_scores = [], [], []

    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            probs = torch.softmax(model(xb), dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_scores.extend(probs[:, 0].cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_scores = np.array(all_scores)

    acc = accuracy_score(all_labels, all_preds)
    p   = precision_score(all_labels, all_preds, zero_division=0)
    r   = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)

    # NaN guard — if model exploded, return worst possible EER
    if np.any(np.isnan(all_scores)) or np.any(np.isinf(all_scores)):
        print('[!] NaN/Inf detected in scores — trial returning EER=50.0')
        return 50.0, None

    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=0)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))] * 100

    if show_plots:
        print(f"\n{'='*10} {dataset_name} Results {'='*10}")
        print(f"EER:       {eer:.4f}%")
        print(f"Accuracy:  {acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"Precision: {p:.4f}")
        print(f"Recall:    {r:.4f}")
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds,
                                    target_names=['Bonafide', 'Spoof']))
        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Bonafide', 'Spoof'],
                    yticklabels=['Bonafide', 'Spoof'])
        ax.set_title(f'Confusion Matrix: {dataset_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()
        return eer, fig

    return eer, None

In [ ]:
# ── Cell 9: Training loop ──────────────────────────────────────────────────────
def training_loop(epochs, model, loss_fn, opt, train_dl, dev_dl,
                  device, scheduler=None, patience=20, trial=None):
    """
    Returns best dev EER achieved.
    Pass `trial` to enable Optuna pruning.
    """
    best_dev_eer = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    freq_mask = T.FrequencyMasking(freq_mask_param=27).to(device)
    time_mask = T.TimeMasking(time_mask_param=35).to(device)

    print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Dev EER':<10} | {'LR':<12}")
    print('-' * 52)

    for epoch in range(epochs):
        # ── Training pass ──────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0

        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            xb = freq_mask(xb)
            xb = time_mask(xb)

            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            running_loss += loss.item()

        if scheduler:
            scheduler.step()

        # ── Evaluation ───────────────────────────
        current_dev_eer, _ = eval_model(model, dev_dl, device, show_plots=False)
        avg_loss   = running_loss / len(train_dl)
        current_lr = opt.param_groups[0]['lr']

        print(f"[{epoch:02d}]   | {avg_loss:.4f}       | "
              f"{current_dev_eer:.2f}%     | {current_lr:.6f}")

        # ── Optuna: report + prune check ───────────────────────────────────────
        if trial is not None:
            trial.report(current_dev_eer, epoch)
            if trial.should_prune():
                print(f'[!] Trial pruned at epoch {epoch}')
                raise optuna.exceptions.TrialPruned()

        # ── Early stopping ─────────────────────────────────────────────────────
        if current_dev_eer < best_dev_eer:
            best_dev_eer = current_dev_eer
            epochs_without_improvement = 0
            best_model_state = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\n[!] Early stopping triggered at epoch {epoch}.')
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return best_dev_eer

In [ ]:
# ── Cell 10: Optuna objective ──────────────────────────────────────────────────
def make_dataloaders(batch_size):
    train_ds = TensorDataset(X_train_tr, y_train_tr)
    dev_ds   = TensorDataset(X_dev_tr,   y_dev_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          pin_memory=True, num_workers=2)
    dev_dl   = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                          pin_memory=True, num_workers=0)
    return train_dl, dev_dl

def objective(trial):
    # ── Search space ───────────────────────────────────────────────────────────
    batch_size      = trial.suggest_categorical('batch_size',      [128, 256, 512])
    lr              = trial.suggest_float('lr',                    1e-5, 1e-3,  log=True)
    weight_decay    = trial.suggest_float('weight_decay',          1e-6, 1e-3,  log=True)
    cnn_dropout     = trial.suggest_float('cnn_dropout',           0.1,  0.4)
    rnn_hidden_size = trial.suggest_categorical('rnn_hidden_size', [64, 128, 256])
    rnn_num_layers  = trial.suggest_int('rnn_num_layers',          1,    3)
    rnn_dropout     = trial.suggest_float('rnn_dropout',           0.0,  0.4)
    bidirectional   = trial.suggest_categorical('bidirectional',   [True, False])
    max_lr_factor   = trial.suggest_float('max_lr_factor',         2.0,  5.0)

    set_seed(22)
    train_dl, dev_dl = make_dataloaders(batch_size)

    model = Deepfake_CRNN(
        x_means, x_stds,
        cnn_dropout=cnn_dropout,
        rnn_hidden_size=rnn_hidden_size,
        rnn_num_layers=rnn_num_layers,
        rnn_dropout=rnn_dropout,
        bidirectional=bidirectional,
    ).to(device)

    weights = torch.tensor([4.0, 1.0], dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.2)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    #scheduler = torch.optim.lr_scheduler.OneCycleLR(
     # opt,
      #max_lr=lr * max_lr_factor,
      #steps_per_epoch=1,
      #epochs=51,
      #pct_start=0.2,
      #div_factor=10,
      #final_div_factor=100
  #)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=50,
    eta_min=1e-6
  )

    best_eer = training_loop(
        epochs=70,
        model=model,
        loss_fn=loss_fn,
        opt=opt,
        train_dl=train_dl,
        dev_dl=dev_dl,
        device=device,
        scheduler=scheduler,
        patience=15,
        trial=trial,
    )
    return best_eer

In [ ]:
# ── Cell 11: Run Optuna study ──────────────────────────────────────────────────
STUDY_DB   = 'sqlite:////content/drive/MyDrive/optuna_crnn.db'
STUDY_NAME = 'crnn_asvspoof'

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    load_if_exists=True,
    direction='minimize',
    sampler=TPESampler(seed=22, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)

study.optimize(
    objective,
    n_trials=40,
    timeout=3 * 3600,
    gc_after_trial=True,
)

print('\n── Optuna complete ──────────────────────────')
print(f'Best EER   : {study.best_value:.4f}%')
print(f'Best params: {study.best_params}')

In [ ]:
# ── Cell 12: Final training with best hyperparameters ─────────────────────────
best = study.best_params
print('Training final model with:', best)

EPOCHS = 200

set_seed(22)
train_dl, dev_dl = make_dataloaders(best['batch_size'])
eval_ds  = TensorDataset(X_eval_tr, y_eval_tr)
eval_dl  = DataLoader(eval_ds, batch_size=best['batch_size'],
                      shuffle=False, pin_memory=True, num_workers=0)

model = Deepfake_CRNN(
    x_means, x_stds,
    cnn_dropout=best['cnn_dropout'],
    rnn_hidden_size=best['rnn_hidden_size'],
    rnn_num_layers=best['rnn_num_layers'],
    rnn_dropout=best['rnn_dropout'],
    bidirectional=best['bidirectional'],
).to(device)

weights = torch.tensor([4.0, 1.0], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.2)

opt = torch.optim.Adam(model.parameters(),
                       lr=best['lr'],
                       weight_decay=best['weight_decay'])

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=best['lr'] * best['max_lr_factor'],
    steps_per_epoch=1,
    epochs=EPOCHS,
    pct_start=0.2,
    div_factor=10,
    final_div_factor=100
)

##scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ##opt,
    ##T_max=EPOCHS,
    ##eta_min=1e-6
##)

print(model)

training_loop(
    epochs=EPOCHS,
    model=model,
    loss_fn=loss_fn,
    opt=opt,
    train_dl=train_dl,
    dev_dl=dev_dl,
    device=device,
    scheduler=scheduler,
    patience=20,
    trial=None,
)

In [ ]:
# ── Cell 13: Evaluate on Dev set ──────────────────────
dev_eer, dev_fig = eval_model(model, dev_dl, device, 'Development Set', show_plots=True)


========== Development Set Results ==========
EER:       0.0179%
Accuracy:  0.9994
F1 Score:  0.9997
Precision: 0.9994
Recall:    1.0000

Classification Report:
              precision    recall  f1-score   support

    Bonafide       1.00      0.99      1.00      2548
       Spoof       1.00      1.00      1.00     22296

    accuracy                           1.00     24844
   macro avg       1.00      1.00      1.00     24844
weighted avg       1.00      1.00      1.00     24844



In [ ]:
# ── Cell 14: Evaluate on Eval set ─────────────────────
eval_eer, eval_fig = eval_model(model, eval_dl, device, 'Evaluation Set', show_plots=True)

print(f'\nGeneralization gap: {eval_eer - dev_eer:.2f}%')


========== Evaluation Set Results ==========
EER:       8.0132%
Accuracy:  0.7638
F1 Score:  0.8484
Precision: 0.9993
Recall:    0.7371

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.30      1.00      0.47      7355
       Spoof       1.00      0.74      0.85     63882

    accuracy                           0.76     71237
   macro avg       0.65      0.87      0.66     71237
weighted avg       0.93      0.76      0.81     71237


Generalization gap: 8.00%


In [ ]:
# ── Cell 15: Save model ────────────────────────────────────────────────────────
torch.save(model.state_dict(), '/content/drive/MyDrive/crnn_best.pth')
torch.save({'means': x_means, 'stds': x_stds},
           '/content/drive/MyDrive/crnn_norm_stats.pth')
print('Model and normalization stats saved to Drive.')

In [ ]:
# ── Cell 16: Reload model ──────────────────────────────────────────────────────
norm_stats = torch.load('/content/drive/MyDrive/crnn_norm_stats.pth')
model = Deepfake_CRNN(
    norm_stats['means'], norm_stats['stds'],
    **{k: best[k] for k in
       ['cnn_dropout','rnn_hidden_size','rnn_num_layers','rnn_dropout','bidirectional']}
).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/crnn_best.pth'))
model.eval()
print('Model reloaded.')